# Etapa 2 — Geração do dataset

Lê os `.npy` da etapa 1, empilha tudo num array único e salva pronto para o treinamento.

Saída:
```
X.npy   shape: (N, 30, 258)  — sequências de landmarks
y.npy   shape: (N,)          — labels inteiros (0, 1, 2...)
```

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt

In [ ]:
SINAIS             = ['Oi', 'Tudo_bem', 'Tchau']
PASTA_DADOS        = 'dados_libras'
FRAMES_POR_AMOSTRA = 30
DIM_LANDMARKS      = 258

# mapeia nome do sinal -> inteiro
label_map = {sinal: i for i, sinal in enumerate(SINAIS)}
print(label_map)

In [ ]:
X, y = [], []
problemas = []

for sinal in SINAIS:
    pasta    = os.path.join(PASTA_DADOS, sinal)
    arquivos = sorted(f for f in os.listdir(pasta) if f.endswith('.npy'))

    for arq in arquivos:
        dados = np.load(os.path.join(pasta, arq))

        if dados.shape != (FRAMES_POR_AMOSTRA, DIM_LANDMARKS):
            problemas.append(f'{sinal}/{arq}: shape errado {dados.shape}')
            continue

        X.append(dados)
        y.append(label_map[sinal])

X = np.array(X, dtype=np.float32)  # (N, 30, 258)
y = np.array(y, dtype=np.int32)    # (N,)

print(f'X: {X.shape}')
print(f'y: {y.shape}')

if problemas:
    print(f'\narquivos ignorados:')
    for p in problemas:
        print(f'  {p}')

In [ ]:
# distribuicao das classes
print('amostras por sinal:')
for sinal, idx in label_map.items():
    n   = (y == idx).sum()
    bar = '█' * n
    print(f'  {sinal:<14} {n:3d}  {bar}')

# proporcao de zeros (landmarks nao detectados)
pct_zeros = (X == 0).mean() * 100
print(f'\nzeros no dataset: {pct_zeros:.1f}%')
if pct_zeros > 40:
    print('aviso: muitos zeros — mediapipe nao detectou bem em varias amostras')
    print('considere regravar as amostras com melhor iluminacao e enquadramento')

In [ ]:
# visualiza a variacao dos landmarks ao longo dos frames para um sinal
# util pra confirmar que o movimento foi capturado (nao e tudo zero)

fig, axes = plt.subplots(1, len(SINAIS), figsize=(14, 3), sharey=False)

for ax, (sinal, idx) in zip(axes, label_map.items()):
    amostras_sinal = X[y == idx]  # todas as amostras desse sinal

    # plota o desvio padrao por frame — mostra onde ha mais movimento
    std_por_frame = amostras_sinal.std(axis=0).mean(axis=1)  # (30,)
    ax.plot(std_por_frame, color='steelblue', linewidth=1.5)
    ax.fill_between(range(30), std_por_frame, alpha=0.2, color='steelblue')
    ax.set_title(sinal.replace('_', ' '), fontsize=11)
    ax.set_xlabel('frame')
    ax.set_ylabel('desvio padrao medio')
    ax.grid(alpha=0.3)

plt.suptitle('variacao dos landmarks por frame (maior = mais movimento)', fontsize=11)
plt.tight_layout()
plt.savefig('variacao_landmarks.png', dpi=120)
plt.show()

In [ ]:
np.save('X.npy', X)
np.save('y.npy', y)

print(f'X.npy salvo  {X.nbytes / 1e6:.1f} MB')
print(f'y.npy salvo  {y.nbytes / 1e3:.1f} KB')